# Nexus-Steg: Training Notebook

**Robust Semantic-Texture Hybrid Steganography System**

This notebook sets up the full training pipeline on Google Colab:

1. Mount Google Drive (checkpoints survive disconnects)
2. Clone the repo and install dependencies
3. Download COCO covers (~46,000 images: val2017 + test2017)
4. Download SpaceNet 2 secrets from 4 cities (~10,000 satellite images)
5. Sanity check, overfit test, full training, evaluation

**Runtime:** Use GPU runtime (A100 recommended). Go to Runtime > Change runtime type > A100.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Clone Repo + Install Dependencies

Clones the `improvement/mod_training_pipeline` branch into Google Drive so all files persist across sessions.

In [ ]:
import os

PROJECT_DIR = "/content/drive/MyDrive/DL_Project/Nexus-Steg"
REPO_URL = "https://github.com/Henildiyora/Nexus-Steg.git"
BRANCH = "improvement/mod_training_pipeline"

if not os.path.exists(PROJECT_DIR):
    os.makedirs(os.path.dirname(PROJECT_DIR), exist_ok=True)
    !git clone -b {BRANCH} {REPO_URL} "{PROJECT_DIR}"
else:
    print(f"Project already exists at {PROJECT_DIR}")
    %cd "{PROJECT_DIR}"
    !git pull origin {BRANCH}

%cd "{PROJECT_DIR}"
!pip install -e . -q
print("\nSetup complete.")

## 3. Download COCO Covers (~7 GB, ~46,000 images)

Downloads COCO val2017 (5,000 images) + test2017 (~41,000 images) as cover images. These are the "innocent" images that will carry hidden secrets. Combined this gives ~46,000 covers -- more than enough diversity.

In [ ]:
import os, glob

%cd "{PROJECT_DIR}"
COVER_DIR = "datasets/cover"
os.makedirs(COVER_DIR, exist_ok=True)

existing_covers = glob.glob(os.path.join(COVER_DIR, "*.[jp][pn][g]*"))
if len(existing_covers) >= 10000:
    print(f"Covers already downloaded: {len(existing_covers)} images found. Skipping.")
else:
    COCO_SETS = {
        "val2017":  "http://images.cocodataset.org/zips/val2017.zip",   # ~1 GB, 5,000 images
        "test2017": "http://images.cocodataset.org/zips/test2017.zip",  # ~6.2 GB, 40,670 images
    }
    for name, url in COCO_SETS.items():
        zip_path = f"/content/{name}.zip"
        if os.path.exists(zip_path):
            print(f"{name}.zip already exists, skipping download.")
        else:
            print(f"Downloading COCO {name}...")
            !wget -q --show-progress {url} -O {zip_path}
        print(f"Extracting {name}...")
        !unzip -q -o {zip_path} -d /content/coco_tmp
        !mv /content/coco_tmp/{name}/* "{COVER_DIR}/"
        !rm -rf /content/coco_tmp {zip_path}
        count = len(glob.glob(os.path.join(COVER_DIR, "*.[jp][pn][g]*")))
        print(f"  Running total: {count} cover images\n")

    total = len(glob.glob(os.path.join(COVER_DIR, "*.[jp][pn][g]*")))
    print(f"Done. {total} cover images in {COVER_DIR}")

## 4. Download SpaceNet 2 Secrets (4 AOIs, ~10,000 satellite images)

Downloads MUL-PanSharpen TIFF images from SpaceNet 2 via direct HTTPS (no AWS CLI or account needed).

| AOI | City | ~Images | Tarball |
|-----|------|---------|--------|
| AOI 2 | Las Vegas | 3,850 | 23 GB |
| AOI 3 | Paris | 1,148 | 5.3 GB |
| AOI 4 | Shanghai | 4,000 | 23.4 GB |
| AOI 5 | Khartoum | 1,000 | 4.7 GB |

**Edit `AOIS_TO_DOWNLOAD` below if disk space is limited** (e.g., just Vegas + Paris for ~5,000 images).

Each tarball is downloaded to local Colab disk, MUL-PanSharpen TIFFs are extracted and moved to Drive, then the tarball is deleted to free space.

In [ ]:
import os, glob, shutil

%cd "{PROJECT_DIR}"

# ====== EDIT THIS to control which AOIs to download ======
AOIS_TO_DOWNLOAD = ["Vegas", "Paris", "Shanghai", "Khartoum"]
# Examples:
#   ["Vegas", "Paris"]             ~5,000 images, ~28 GB download
#   ["Vegas", "Paris", "Khartoum"] ~6,000 images, ~33 GB download
#   ["Vegas", "Paris", "Shanghai", "Khartoum"]  ~10,000 images, ~56 GB download
# =========================================================

HTTPS_BASE = "https://spacenet-dataset.s3.amazonaws.com/spacenet/SN2_buildings/tarballs"
AOI_MAP = {
    "Vegas":    {"file": "SN2_buildings_train_AOI_2_Vegas.tar.gz",    "num": 2},
    "Paris":    {"file": "SN2_buildings_train_AOI_3_Paris.tar.gz",    "num": 3},
    "Shanghai": {"file": "SN2_buildings_train_AOI_4_Shanghai.tar.gz", "num": 4},
    "Khartoum": {"file": "SN2_buildings_train_AOI_5_Khartoum.tar.gz", "num": 5},
}

SECRET_DIR = os.path.join(PROJECT_DIR, "datasets/secret/MUL-PanSharpen")
os.makedirs(SECRET_DIR, exist_ok=True)

existing_secrets = glob.glob(os.path.join(SECRET_DIR, "*.tif"))
if len(existing_secrets) >= 5000:
    print(f"Secrets already downloaded: {len(existing_secrets)} images found. Skipping.")
else:
    for city in AOIS_TO_DOWNLOAD:
        info = AOI_MAP[city]
        tarball = info["file"]
        aoi_num = info["num"]
        local_tar = f"/content/{tarball}"
        extract_dir = "/content/spacenet_tmp"

        print(f"\n{'='*60}")
        print(f"Downloading AOI {aoi_num} ({city})...")
        print(f"{'='*60}")

        url = f"{HTTPS_BASE}/{tarball}"
        !wget -q --show-progress "{url}" -O "{local_tar}"

        if not os.path.exists(local_tar) or os.path.getsize(local_tar) < 1000:
            print(f"  WARNING: Download failed for {city}. Skipping.")
            !rm -f "{local_tar}"
            continue

        print(f"Extracting MUL-PanSharpen images...")
        os.makedirs(extract_dir, exist_ok=True)
        !tar xzf "{local_tar}" -C "{extract_dir}" --wildcards '*/MUL-PanSharpen/*.tif'

        tif_files = glob.glob(os.path.join(extract_dir, "**", "MUL-PanSharpen", "*.tif"), recursive=True)
        print(f"Found {len(tif_files)} TIFF files from {city}")

        for f in tif_files:
            dest = os.path.join(SECRET_DIR, os.path.basename(f))
            if not os.path.exists(dest):
                shutil.move(f, dest)

        print(f"Cleaning up tarball and temp files...")
        !rm -rf "{local_tar}" "{extract_dir}"

    total = len(glob.glob(os.path.join(SECRET_DIR, "*.tif")))
    print(f"\nDone. Total secret images: {total}")

## 5. Verify Dataset

Check that both folders have images and display a few samples.

In [ ]:
import os, glob
import matplotlib.pyplot as plt
from PIL import Image
import numpy as np

%cd "{PROJECT_DIR}"

cover_files = sorted(glob.glob("datasets/cover/*.[jp][pn][g]*"))
secret_files = sorted(glob.glob("datasets/secret/MUL-PanSharpen/*.tif"))

print(f"Cover images:  {len(cover_files)}")
print(f"Secret images: {len(secret_files)}")
print(f"Training pairs (min): {min(len(cover_files), len(secret_files))}")

assert len(cover_files) > 0, "No cover images found!"
assert len(secret_files) > 0, "No secret images found!"

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
fig.suptitle("Dataset Samples (top: covers, bottom: secrets)", fontsize=14)

for i in range(4):
    img = Image.open(cover_files[i * len(cover_files) // 4]).convert("RGB")
    axes[0, i].imshow(img)
    axes[0, i].set_title(f"Cover {i+1}")
    axes[0, i].axis("off")

import tifffile as tiff
for i in range(4):
    raw = tiff.imread(secret_files[i * len(secret_files) // 4])
    if raw.dtype == np.uint16:
        p2, p98 = np.percentile(raw, (2, 98))
        raw = np.clip(raw, p2, p98)
        raw = ((raw - p2) / max(p98 - p2, 1e-6) * 255).astype(np.uint8)
    if len(raw.shape) == 3:
        if raw.shape[0] < raw.shape[2]:
            raw = raw.transpose(1, 2, 0)
        raw = raw[:, :, :3]
    axes[1, i].imshow(raw)
    axes[1, i].set_title(f"Secret {i+1}")
    axes[1, i].axis("off")

plt.tight_layout()
plt.show()

## 6. Sanity Check

Verifies that initial losses match expected values before training:

| Loss | Expected | If Wrong |
|------|----------|----------|
| `l_inv` | 0.01 - 0.10 | Encoder init problem |
| `l_rec` | 0.30 - 0.70 | Reveal network problem |
| `l_disc` | ~0.693 | Discriminator init problem |

Also saves `results/sanity_inputs.png` -- open it to verify images look correct.

In [ ]:
%cd "{PROJECT_DIR}"
!python main.py --sanity

## 7. Overfit One Batch

Trains on a single batch for 200 steps. Verifies the model has enough capacity to memorize one batch.

- **PASS** (loss < 0.01): Model capacity is sufficient.
- **WARN** (loss 0.01-0.10): Probably fine, monitor during training.
- **FAIL** (loss > 0.10): Architecture or learning rate problem. Do not proceed to full training.

In [ ]:
%cd "{PROJECT_DIR}"
!python main.py --overfit_one_batch

## 8. Train

Full 100-epoch training with 3-phase curriculum:

| Phase | Epochs | What Happens |
|-------|--------|-------------|
| 1 | 0-29 | Pure hiding/recovery. No noise, no adversarial. |
| 2 | 30-59 | Noise layer ON, adversarial ON (mild). PSNR(secret) will drop at epoch 30 then recover. |
| 3 | 60-99 | Full adversarial pressure. Metrics should plateau. |

**Early stopping:** Training auto-stops if PSNR(secret) doesn't improve for 15 epochs.

**Best checkpoint** is auto-saved to `checkpoints/nexus_best.pth`.

**Training log** is written to `results/training_log.csv` every epoch.

In [ ]:
%cd "{PROJECT_DIR}"
!python main.py --epochs 100 --batch_size 64 --checkpoint_every 10 --patience 15

## 9. Plot Training Curves

Reads `results/training_log.csv` and plots PSNR, SSIM, and losses over epochs. Vertical dashed lines mark phase transitions (epoch 30, 60).

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

%cd "{PROJECT_DIR}"
df = pd.read_csv("results/training_log.csv")

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle("Nexus-Steg Training Curves", fontsize=16)

# PSNR
ax = axes[0, 0]
ax.plot(df["epoch"], df["val_psnr_stego"], label="PSNR(stego)", linewidth=2)
ax.plot(df["epoch"], df["val_psnr_secret"], label="PSNR(secret)", linewidth=2)
ax.axvline(x=30, color="gray", linestyle="--", alpha=0.5, label="Phase 2")
ax.axvline(x=60, color="gray", linestyle=":", alpha=0.5, label="Phase 3")
ax.set_xlabel("Epoch")
ax.set_ylabel("PSNR (dB)")
ax.set_title("Validation PSNR")
ax.legend()
ax.grid(True, alpha=0.3)

# SSIM
ax = axes[0, 1]
ax.plot(df["epoch"], df["val_ssim_stego"], label="SSIM(stego)", linewidth=2)
ax.plot(df["epoch"], df["val_ssim_secret"], label="SSIM(secret)", linewidth=2)
ax.axvline(x=30, color="gray", linestyle="--", alpha=0.5)
ax.axvline(x=60, color="gray", linestyle=":", alpha=0.5)
ax.set_xlabel("Epoch")
ax.set_ylabel("SSIM")
ax.set_title("Validation SSIM")
ax.legend()
ax.grid(True, alpha=0.3)

# Training losses
ax = axes[1, 0]
ax.plot(df["epoch"], df["l_inv"], label="l_inv (invisibility)", linewidth=1.5)
ax.plot(df["epoch"], df["l_rec"], label="l_rec (recovery)", linewidth=1.5)
ax.plot(df["epoch"], df["l_disc"], label="l_disc (discriminator)", linewidth=1.5)
ax.axvline(x=30, color="gray", linestyle="--", alpha=0.5)
ax.axvline(x=60, color="gray", linestyle=":", alpha=0.5)
ax.set_xlabel("Epoch")
ax.set_ylabel("Loss")
ax.set_title("Training Losses")
ax.legend()
ax.grid(True, alpha=0.3)

# Learning rate
ax = axes[1, 1]
ax.plot(df["epoch"], df["lr"], linewidth=2, color="tab:orange")
ax.axvline(x=30, color="gray", linestyle="--", alpha=0.5)
ax.axvline(x=60, color="gray", linestyle=":", alpha=0.5)
ax.set_xlabel("Epoch")
ax.set_ylabel("Learning Rate")
ax.set_title("Cosine Annealing LR")
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("results/training_curves.png", dpi=150)
plt.show()
print("Saved to results/training_curves.png")

## 10. Evaluate

Runs 8 attack tests against the best checkpoint:

1. Basic Recovery (no attack)
2. JPEG Q=90
3. JPEG Q=50 (heavy)
4. Gaussian Blur (sigma=2.0)
5. Gaussian Noise (std=0.05)
6. Screenshot Simulation (resize 50%)
7. Social Media (resize 75% + JPEG 70)
8. Steganalysis Detection

In [ ]:
%cd "{PROJECT_DIR}"
!python evaluate.py --checkpoint checkpoints/nexus_best.pth

## 11. Display Visual Results

Shows training progression (one sample from each phase) and evaluation attack strips.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import glob, os

%cd "{PROJECT_DIR}"

epoch_imgs = []
for target in [0, 29, 59, 99]:
    path = f"results/epoch_{target}.png"
    if os.path.exists(path):
        epoch_imgs.append((target, path))

if not epoch_imgs:
    all_epochs = sorted(glob.glob("results/epoch_*.png"))
    for p in all_epochs[:4]:
        e = int(os.path.basename(p).split("_")[1].split(".")[0])
        epoch_imgs.append((e, p))

if epoch_imgs:
    fig, axes = plt.subplots(len(epoch_imgs), 1, figsize=(20, 5 * len(epoch_imgs)))
    if len(epoch_imgs) == 1:
        axes = [axes]
    for ax, (e, path) in zip(axes, epoch_imgs):
        img = mpimg.imread(path)
        ax.imshow(img)
        ax.set_title(f"Epoch {e}  [cover | secret | stego | revealed]", fontsize=13)
        ax.axis("off")
    plt.tight_layout()
    plt.show()
else:
    print("No epoch images found yet.")

print("\n--- Evaluation Attack Results ---\n")
eval_imgs = sorted(glob.glob("results/evaluation/*.png"))
if eval_imgs:
    fig, axes = plt.subplots(len(eval_imgs), 1, figsize=(20, 4 * len(eval_imgs)))
    if len(eval_imgs) == 1:
        axes = [axes]
    for ax, path in zip(axes, eval_imgs):
        img = mpimg.imread(path)
        name = os.path.splitext(os.path.basename(path))[0].replace("_", " ").title()
        ax.imshow(img)
        ax.set_title(name, fontsize=12)
        ax.axis("off")
    plt.tight_layout()
    plt.show()
else:
    print("No evaluation images found. Run Cell 10 first.")

## 12. Evaluation Report

Prints the full PASS/WARN/FAIL report from the evaluation test suite.

In [ ]:
import os

%cd "{PROJECT_DIR}"
report_path = "results/evaluation/report.txt"

if os.path.exists(report_path):
    with open(report_path) as f:
        print(f.read())
else:
    print("No evaluation report found. Run Cell 10 first.")